# 第14回　コンピューターによるデータ解析入門（2）

今回は、前回に続いて海水温データを使い、**時間とともに変化するデータをどのように見るか**を学びます。

主に扱う内容は、

- 移動平均
- 変動の大きさ（標準偏差）
- 周期的な変動（調和解析）

です。

前回は、最小二乗法を使ってデータの**変化傾向**を調べました。今回は、平均的な傾向だけではなく、そのまわりで起こる**変動**に注目します。

> **このNotebookのCodeセルは、基本的に上から順番に実行してください。**  
> 前のセルで作成した変数やデータを、後のセルでも使用します。  
> Codeセルは **Shift + Enter** で実行できます。

## 1. 前回の復習

前回は、気象庁の沿岸域の海水温データを読み込み、日付を作成し、年平均水温を計算しました。

今回も豊後水道南部の `area518.txt` を使います。ファイルがNotebookと同じフォルダにあることを確認してください。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("area518.txt")
df["date"] = pd.to_datetime(dict(year=df["yyyy"], month=df["mm"], day=df["dd"]))

annual = df.groupby("yyyy")["Temp."].mean()

plt.plot(annual.index, annual.values, "-o")
plt.xlabel("Year")
plt.ylabel("Annual mean temperature")
plt.show()

前回の最小二乗法では、年平均水温を

$$
y \simeq ax+b
$$

という直線で近似し、長期的な変化傾向を調べました。

しかし、年平均水温の図を見ると、長期的な変化だけでなく、年ごとの上下もあります。今回は、このような**時間変動**を見ていきます。

## 2. 移動平均

年平均水温には、長期的な変化と年ごとの変動が重なっています。

長い時間スケールの変化を見やすくする方法の一つが**移動平均**です。

ここでは5年移動平均を考えます。ある年を中心とした5年間の値を平均し、その値をその年の移動平均とします。

In [ ]:
annual_ma5 = annual.rolling(5, center=True).mean()

plt.plot(annual.index, annual.values, "-o", label="Annual mean")
plt.plot(annual_ma5.index, annual_ma5.values, linewidth=2, label="5-year moving average")
plt.xlabel("Year")
plt.ylabel("Temperature")
plt.legend()
plt.show()

`rolling(5, center=True)` は、5個のデータを1つの窓として、その窓を少しずつ動かします。

`center=True` とすることで、5年間の平均値をその5年間の中央の年に対応させています。

5年移動平均では、年ごとの細かな変動が小さくなり、より長い時間スケールの変化が見やすくなります。

### 5年以上の変動と5年以下の変動

5年移動平均を、ここでは**5年以上の時間スケールの変動**とみなします。

一方、

**年平均水温 − 5年移動平均**

を計算すると、移動平均では表現されなかった、より短い時間スケールの変動を見ることができます。

In [ ]:
long_variation = annual_ma5
short_variation = annual - annual_ma5

plt.plot(short_variation.index, short_variation.values, "-o")
plt.axhline(0, linewidth=1)
plt.xlabel("Year")
plt.ylabel("Annual mean - 5-year moving average")
plt.show()

5年移動平均は、どの時間スケールを「長い変動」「短い変動」と考えるかを決める一つの方法です。

ここでは授業の例として5年を使います。

### 課題1へ

ここまでで、年平均水温に5年移動平均を適用し、5年以上・5年以下の時間スケールの変動を分けて考えられるようになりました。

`exercise14.ipynb` の **課題1** に進んでください。

課題1が終わったら、このNotebookに戻ってください。

## 3. 日平均水温にも移動平均を使う

移動平均は年平均データだけでなく、日平均データにも使えます。

日平均水温には季節変化や日々の変動が含まれています。ここでは365日移動平均を適用してみます。

In [ ]:
daily = df.set_index("date")["Temp."]
daily_ma365 = daily.rolling(365, center=True).mean()

plt.plot(daily.index, daily.values, linewidth=0.6, label="Daily")
plt.plot(daily_ma365.index, daily_ma365.values, linewidth=2, label="365-day moving average")
plt.xlabel("Date")
plt.ylabel("Temperature")
plt.legend()
plt.show()

365日移動平均を使うと、日々の変動や季節変化が大きくならされ、より長い時間スケールの変化が見やすくなります。

### 課題2へ

`exercise14.ipynb` の **課題2** に進んでください。

課題2が終わったら、このNotebookに戻り、次の「変動の大きさ」へ進みます。

## 4. 変動の大きさを数値で表す

これまでは、時系列を図にして変動を見てきました。

しかし、「どちらの変動が大きいか」を客観的に比較するには、変動の大きさを**数値**で表す必要があります。

そのために使う代表的な量が**標準偏差**です。

データ $y_i$ の平均を $\bar y$ とし、

$$
y_i'=y_i-\bar y
$$

とすると、平均からのずれの大きさは

$$
\sqrt{
\frac{1}{N}
\sum_{i=1}^{N}{y_i'}^2
}
$$

で表せます。

これは、平均からのずれを2乗して平均し、最後に平方根をとったものです。

### 実際に計算してみる

まず、豊後水道南部の年平均水温について、標準偏差を式どおりに計算してみます。

In [ ]:
y = annual.dropna().to_numpy()
ya = y - y.mean()

std_manual = np.sqrt(np.sum(ya**2) / len(ya))
print("式から計算 =", std_manual)

print("NumPy（ddof=0） =", np.std(y, ddof=0))
print("pandas（標本標準偏差） =", annual.std())

NumPyの `np.std(y, ddof=0)` は、ここで使った

$$
\sqrt{\frac{1}{N}\sum_i(y_i-\bar y)^2}
$$

を計算します。

一方、pandasの `Series.std()` は標準では分母に $N-1$ を使います。授業ではまず、変動の大きさの意味が分かりやすい $N$ で割る式を使います。

### 3種類の変動を比較する

次の3つについて、標準偏差を比較します。

1. 年平均水温の変動
2. 5年移動平均で表される、5年以上の時間スケールの変動
3. 年平均水温から5年移動平均を引いた、5年以下の時間スケールの変動

In [ ]:
annual_valid = annual.dropna()
long_valid = long_variation.dropna()
short_valid = short_variation.dropna()

std_annual = np.std(annual_valid.values, ddof=0)
std_long = np.std(long_valid.values, ddof=0)
std_short = np.std(short_valid.values, ddof=0)

print("年平均水温                 :", std_annual)
print("5年以上の時間スケールの変動:", std_long)
print("5年以下の時間スケールの変動:", std_short)

標準偏差を使うことで、図を見た印象だけではなく、それぞれの変動の大きさを数値として比較できます。

### 課題3へ

ここまでで、標準偏差を使って変動の大きさを数値化できるようになりました。

`exercise14.ipynb` の **課題3** に進んでください。

課題3が終わったら、このNotebookに戻り、次の「周期的な変動」へ進みます。

## 5. 周期的な変動

時系列には、ある一定の周期で繰り返す変動が含まれることがあります。

周期 $T$ の変動は、sin関数とcos関数を使って

$$
\sin\left(\frac{2\pi x'}{T}\right),
\qquad
\cos\left(\frac{2\pi x'}{T}\right)
$$

と表すことができます。

ここでは、昨年度の例と同じく**22年周期**の変動を考えます。

まず、年を平均からの偏差

$$
x_i'=x_i-\bar x
$$

にします。

In [ ]:
x = annual.index.to_numpy(dtype=float)
f = annual.to_numpy(dtype=float)

mask = np.isfinite(f)
x = x[mask]
f = f[mask]

xp = x - x.mean()

T = 22.0
sin22 = np.sin(2*np.pi*xp/T)
cos22 = np.cos(2*np.pi*xp/T)

plt.plot(xp, sin22, label="sin")
plt.plot(xp, cos22, label="cos")
plt.xlabel("Year anomaly")
plt.legend()
plt.show()

22年周期の変動を

$$
f_i \simeq
A\sin\left(\frac{2\pi x_i'}{22}\right)
+
B\cos\left(\frac{2\pi x_i'}{22}\right)
$$

と表してみます。

ここで $A$ と $B$ を決めれば、22年周期の変動をデータから取り出すことができます。

これは前回の

$$
y\simeq ax+b
$$

とよく似ています。前回は直線をデータに当てはめましたが、今回はsin関数とcos関数をデータに当てはめます。

### sin成分の係数 A

まずsin成分だけを考え、

$$
f_i\simeq A g_i,
\qquad
g_i=\sin\left(\frac{2\pi x_i'}{22}\right)
$$

とします。

前回の最小二乗法と同じ考え方から、

$$
A=
\frac{\sum_i g_i f_i}
{\sum_i g_i^2}
=
\frac{
\sum_i f_i\sin(2\pi x_i'/22)
}{
\sum_i\sin^2(2\pi x_i'/22)
}
$$

となります。

### cos成分の係数 B

同様に、

$$
f_i\simeq B g_i,
\qquad
g_i=\cos\left(\frac{2\pi x_i'}{22}\right)
$$

と考えると、

$$
B=
\frac{
\sum_i f_i\cos(2\pi x_i'/22)
}{
\sum_i\cos^2(2\pi x_i'/22)
}
$$

となります。

In [ ]:
A = np.sum(f * sin22) / np.sum(sin22**2)
B = np.sum(f * cos22) / np.sum(cos22**2)

print("A =", A)
print("B =", B)

求めた `A`, `B` を使って、22年周期の変動を年平均水温に重ねてみます。

In [ ]:
fit22 = A*sin22 + B*cos22

plt.plot(x, f, "-o", label="Annual mean")
plt.plot(x, fit22 + f.mean(), linewidth=2, label="22-year component + mean")
plt.xlabel("Year")
plt.ylabel("Temperature")
plt.legend()
plt.show()

## 6. 周期的な変動の振幅

sin成分とcos成分を合わせた

$$
A\sin\theta+B\cos\theta
$$

は、位相 $\phi$ を使って

$$
\sqrt{A^2+B^2}\sin(\theta+\phi)
$$

と書くことができます。

したがって、22年周期の変動の**振幅**は

$$
\boxed{\sqrt{A^2+B^2}}
$$

です。

In [ ]:
amplitude22 = np.sqrt(A**2 + B**2)
print("22年周期の変動の振幅 =", amplitude22)

### 補足：なぜ振幅が $\sqrt{A^2+B^2}$ になるのか

$$
C\sin(\theta+\phi)
=
C\sin\theta\cos\phi
+
C\cos\theta\sin\phi
$$

なので、

$$
A=C\cos\phi,\qquad B=C\sin\phi
$$

と対応させることができます。

したがって、

$$
A^2+B^2
=
C^2(\cos^2\phi+\sin^2\phi)
=
C^2
$$

より、

$$
C=\sqrt{A^2+B^2}
$$

となります。

### 課題4へ

ここまでで、22年周期のsin・cos成分を求め、その振幅を数値化できるようになりました。

`exercise14.ipynb` の **課題4** に進んでください。

## 7. まとめ

今回は、時系列データに含まれる変動を、

- 移動平均によって時間スケールごとに分ける
- 標準偏差によって変動の大きさを数値化する
- sin・cos関数を当てはめて周期的な変動を取り出す

という方法で調べました。

前回の直線の当てはめと、今回の周期関数の当てはめは、どちらも**データと関数の差を小さくするように係数を決める**という同じ考え方に基づいています。

より多くの周期を一度に扱う方法は、発展教材 `fft.ipynb` で扱います。